# Book Platform Market Analysis with SQL

This notebook analyses a database of books, authors, publishers, ratings and reviews to support the value proposition of a new reading app. Each question below is answered with a single SQL query.

In [1]:
import os
from getpass import getpass

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [2]:
# Connection settings. The password is read from the DB_PASSWORD environment
# variable, or requested at runtime, so it is never stored in the notebook.
db_config = {'user': os.getenv('DB_USER', 'practicum_student'),
             'pwd': os.getenv('DB_PASSWORD') or getpass('Database password: '),
             'host': os.getenv('DB_HOST', 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net'),
             'port': int(os.getenv('DB_PORT', 6432)),
             'db': os.getenv('DB_NAME', 'data-analyst-final-project-db')}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                         db_config['pwd'],
                                                         db_config['host'],
                                                         db_config['port'],
                                                         db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [3]:
with engine.connect() as con:
    f = con.execute(text("""SELECT * 
                    FROM books 
                    ORDER BY num_pages desc"""))
    df_books = pd.DataFrame(f.fetchall())
    df_books.columns =f.keys()

    df_books

In [4]:
df_books.head()

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,298,236,Harry Potter Boxed Set Books 1-5 (Harry Potte...,2690,2004-09-13,253
1,684,354,The Complete Anne of Green Gables Boxed Set (A...,2088,1998-10-06,285
2,361,240,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,1728,2012-09-25,33
3,760,224,The Iliad/The Odyssey,1556,1999-11-01,217
4,685,57,The Complete Calvin and Hobbes,1456,2005-09-06,18


In [5]:
with engine.connect() as con:
    a = con.execute(text("""SELECT * 
                       FROM authors"""))
    df_authors = pd.DataFrame(a.fetchall())
    df_authors.columns =a.keys()

In [6]:
df_authors.head()

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [7]:
with engine.connect() as con:
    p = con.execute(text("""select * 
                        from publishers"""))
    df_pub = pd.DataFrame(p.fetchall())
    df_pub.columns =p.keys()

In [8]:
df_pub.head()

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [9]:
with engine.connect() as con:
    r = con.execute(text("""SELECT *
                       FROM ratings"""))
    df_rate = pd.DataFrame(r.fetchall())
    df_rate.columns =r.keys()

In [10]:
df_rate.head()

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [11]:
with engine.connect() as con:
    re = con.execute(text("""SELECT * 
                     FROM reviews """))
    df_rev = pd.DataFrame(re.fetchall())
    df_rev.columns =re.keys()

In [12]:
df_rev.head()

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


1. Find the number of books released after January 1, 2000.

In [13]:
with engine.connect() as con:
    bn = con.execute(text("""SELECT count(book_id) 
                     FROM books 
                     WHERE publication_date > '2000-01-01' """))
    df2 = pd.DataFrame(bn.fetchall())
    df2.columns =bn.keys() 
    

In [14]:
df2

,count
0,819


2. Find the number of user reviews and the average rating for each book.

In [15]:
with engine.connect() as con:
    rr = con.execute(text("""SELECT b.book_id,
                                    b.title,
                                    (SELECT COUNT(*) FROM reviews AS rv
                                     WHERE rv.book_id = b.book_id) AS review_count,
                                    (SELECT ROUND(AVG(rating), 2) FROM ratings AS ra
                                     WHERE ra.book_id = b.book_id) AS avg_rating
                             FROM books AS b
                             ORDER BY review_count DESC"""))
    df3 = pd.DataFrame(rr.fetchall())
    df3.columns = rr.keys()

In [16]:
df3

,book_id,title,review_count,avg_rating
0,948,Twilight (Twilight #1),7,3.66
1,963,Water for Elephants,6,3.98
2,734,The Glass Castle,6,4.21
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
4,695,The Curious Incident of the Dog in the Night-Time,6,4.08
...,...,...,...,...
995,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.67
996,808,The Natural Way to Draw,0,3.00
997,672,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.00
998,221,Essential Tales and Poems,0,4.00


3. Identify the publisher that has released the greatest number of books with more than 50 pages (this will help you exclude brochures and similar publications from your analysis).

In [17]:
with engine.connect() as con:
    pb = con.execute(text("""
                     SELECT publisher, count(book_id) as book_num
                     FROM publishers as p  
                     JOIN books as b 
                         ON p.publisher_id = b.publisher_id 
                     WHERE num_pages > 50 
                     GROUP BY p.publisher 
                     ORDER BY count(book_id) desc LIMIT 5 """))
    df4 = pd.DataFrame(pb.fetchall())
    df4.columns =pb.keys() 
    

In [18]:
df4

,publisher,book_num
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


4. Identify the author with the highest average book rating (look only at books with at least 50 ratings).

In [19]:
# Keep only books with at least 50 ratings, then average by author
with engine.connect() as con:
    avb = con.execute(text("""SELECT a.author, ROUND(AVG(r.rating), 2) AS avg_rating
                              FROM ratings AS r
                              JOIN books AS b ON r.book_id = b.book_id
                              JOIN authors AS a ON b.author_id = a.author_id
                              WHERE r.book_id IN (SELECT book_id
                                                  FROM ratings
                                                  GROUP BY book_id
                                                  HAVING COUNT(rating_id) >= 50)
                              GROUP BY a.author
                              ORDER BY avg_rating DESC"""))
    df5 = pd.DataFrame(avb.fetchall())
    df5.columns = avb.keys()

In [20]:
df5

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.29
1,Markus Zusak/Cao Xuân Việt Khương,4.26
2,J.R.R. Tolkien,4.25
3,Louisa May Alcott,4.19
4,Rick Riordan,4.08
5,William Golding,3.90
6,J.D. Salinger,3.83
7,William Shakespeare/Paul Werstine/Barbara A. M...,3.79
8,Paulo Coelho/Alan R. Clarke/Özdemir İnce,3.79
9,Lois Lowry,3.75


5. Find the average number of text reviews among users who rated more than 50 books.

In [21]:
with engine.connect() as con:
    rv = con.execute(text("""SELECT avg(Text_review) AS avg_Treview
                     FROM (SELECT username, Count(text) As Text_review 
                     FROM reviews 
                     GROUP BY username) as c_table 
                     
                     WHERE username IN (SELECT username FROM ratings GROUP BY username HAVING count(username) > 50)"""))
    df6 = pd.DataFrame(rv.fetchall())
    df6.columns =rv.keys() 

In [22]:
df6

,avg_treview
0,24.3333333333333333
